In [1]:
import sys
print(sys.executable)


C:\Users\student\anaconda3\envs\darkflow\python.exe


In [2]:
import sys

sys.path.append(r"C:\Users\student\darkflow")

from darkflow.net.build import TFNet

print("Darkflow loaded successfully")

The TensorFlow contrib module will not be included in TensorFlow 2.0.
For more information, please see:
  * https://github.com/tensorflow/community/blob/master/rfcs/20180907-contrib-sunset.md
  * https://github.com/tensorflow/addons
  * https://github.com/tensorflow/io (for I/O related ops)
If you depend on functionality not listed there, please file an issue.









Darkflow loaded successfully


In [3]:
import os

print(os.path.exists(r"C:\Users\student\darkflow\cfg\yolo.cfg"))
print(os.path.exists(r"C:\Users\student\darkflow\bin\yolov2.weights"))

True
True


In [4]:
options = {
    "model": r"C:\Users\student\darkflow\cfg\yolo.cfg",
    "load": r"C:\Users\student\darkflow\bin\yolov2.weights",
    "threshold": 0.3
}

tfnet = TFNet(options)

Parsing C:\Users\student\darkflow\cfg\yolo.cfg
Loading C:\Users\student\darkflow\bin\yolov2.weights ...
Successfully identified 203934260 bytes
Finished in 0.025924205780029297s
Model has a coco model name, loading coco labels.

Building net ...

Source | Train? | Layer description                | Output size
-------+--------+----------------------------------+---------------



       |        | input                            | (?, 608, 608, 3)
 Load  |  Yep!  | conv 3x3p1_1  +bnorm  leaky      | (?, 608, 608, 32)



C:\Users\student\darkflow\darkflow\dark\darknet.py:54: UserWarning: ./cfg/yolov2.cfg not found, use C:\Users\student\darkflow\cfg\yolo.cfg instead
  cfg_path, FLAGS.model))


 Load  |  Yep!  | maxp 2x2p0_2                     | (?, 304, 304, 32)
 Load  |  Yep!  | conv 3x3p1_1  +bnorm  leaky      | (?, 304, 304, 64)
 Load  |  Yep!  | maxp 2x2p0_2                     | (?, 152, 152, 64)
 Load  |  Yep!  | conv 3x3p1_1  +bnorm  leaky      | (?, 152, 152, 128)
 Load  |  Yep!  | conv 1x1p0_1  +bnorm  leaky      | (?, 152, 152, 64)
 Load  |  Yep!  | conv 3x3p1_1  +bnorm  leaky      | (?, 152, 152, 128)
 Load  |  Yep!  | maxp 2x2p0_2                     | (?, 76, 76, 128)
 Load  |  Yep!  | conv 3x3p1_1  +bnorm  leaky      | (?, 76, 76, 256)
 Load  |  Yep!  | conv 1x1p0_1  +bnorm  leaky      | (?, 76, 76, 128)
 Load  |  Yep!  | conv 3x3p1_1  +bnorm  leaky      | (?, 76, 76, 256)
 Load  |  Yep!  | maxp 2x2p0_2                     | (?, 38, 38, 256)
 Load  |  Yep!  | conv 3x3p1_1  +bnorm  leaky      | (?, 38, 38, 512)
 Load  |  Yep!  | conv 1x1p0_1  +bnorm  leaky      | (?, 38, 38, 256)
 Load  |  Yep!  | conv 3x3p1_1  +bnorm  leaky      | (?, 38, 38, 512)
 Load  |  Ye

In [5]:
import cv2

img = cv2.imread(r"C:\Users\student\darkflow\sample_img\sample_person.jpg")

result = tfnet.return_predict(img)

result

[{'label': 'person',
  'confidence': 0.8181257,
  'topleft': {'x': 189, 'y': 95},
  'bottomright': {'x': 271, 'y': 380}},
 {'label': 'dog',
  'confidence': 0.8000395,
  'topleft': {'x': 69, 'y': 258},
  'bottomright': {'x': 209, 'y': 355}},
 {'label': 'horse',
  'confidence': 0.88891464,
  'topleft': {'x': 397, 'y': 127},
  'bottomright': {'x': 605, 'y': 352}}]

In [ ]:
import cv2

cap = cv2.VideoCapture(r"C:\Users\student\darkflow\testing.mp4.mp4")

while cap.isOpened():
    ret, frame = cap.read()

    if not ret:
        break

    results = tfnet.return_predict(frame)

    for r in results:
        tl = (r['topleft']['x'], r['topleft']['y'])
        br = (r['bottomright']['x'], r['bottomright']['y'])
        label = r['label']

        cv2.rectangle(frame, tl, br, (0,255,0), 2)
        cv2.putText(frame,
                    label,
                    (tl[0], tl[1]-10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (255,255,255),
                    2)

    cv2.imshow("YOLOv2 Detection", frame)

    if cv2.waitKey(1) & 0xFF == 27:  # ESC
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
import cv2

video_path = r"C:\Users\student\darkflow\testing.mp4.mp4"

cap = cv2.VideoCapture(video_path)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

out = cv2.VideoWriter(
    "output_detection.mp4",
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps,
    (width, height)
)

while cap.isOpened():
    ret, frame = cap.read()

    if not ret:
        break

    results = tfnet.return_predict(frame)

    for r in results:
        tl = (r['topleft']['x'], r['topleft']['y'])
        br = (r['bottomright']['x'], r['bottomright']['y'])

        cv2.rectangle(frame, tl, br, (0,255,0), 2)
        cv2.putText(frame,
                    r['label'],
                    (tl[0], tl[1]-10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (255,255,255),
                    2)

    out.write(frame)

cap.release()
out.release()

print("Video saved successfully")